# Operator Creation Tutorial

This tutorial will teach you how to interface with the Operator class in matrices.py to add/remove general operators from the Operators Tutorial.npz object file. The Operators Tutorial.npz object contains a dictionary of operators. The key expresses the name of the operator while the value associated with the key is a complex-valued array. This tutorial will focus on creating and modifying a object file named Operators Tutorial.npz. There exist a copy of the file under the QuDiPy tutorial data directory.

## 1. Add current location path and import packages

In [ ]:
import os, sys
sys.path.append(os.path.dirname(os.getcwd()))

original_dir = os.path.dirname(os.getcwd())
print(original_dir)

# This will be used to access a tutorial object file
tutorial_data_dir = os.path.join(original_dir,'tutorials','QuDiPy tutorial data')

print(tutorial_data_dir)

import qudipy.qutils.matrices as matr

import numpy as np

## 2. Initialize an operator object from the Operator class

If no Operators Tutorials.npz file exist an object, called ops, can be initialized with a hard coded operator dictionary or left as an empty dictionary. For the tutorial examples that follow you will use Operators Tutorial.npz object file which you will  start by creating.

In [ ]:
# Dictionary to initialize ops object with
init_ops = {
    'PAULI_X': np.array([[0, 1], [1, 0]], dtype=complex),
    'PAULI_Y': np.array([[0, -1.0j], 
        [1.0j, 0]],dtype=complex),
    'PAULI_Z': np.array([[1, 0], [0, -1]], dtype=complex),
    'PAULI_I': np.array([[1, 0], [0, 1]], dtype=complex),
    'PAULI_I_4x4': np.array([[1, 0, 0, 0], 
                            [0, 1, 0, 0], 
                            [0, 0, 1, 0], 
                            [0, 0, 0, 1]], dtype=complex)
}

### 2.1 With an existing dictionary but no object file

In [ ]:
#Initialize the operator object library
ops1 = matr.Operator(operators=init_ops)

# Specify filename for operators object
filename =os.path.join(tutorial_data_dir, 'Operators Tutorial.npz')
# or 
# filename =os.path.join(tutorial_data_dir, 'Operators Tutorial')

print(f'File name: {filename}')

# Now you can save the object file to the tutorial data directory using the 
# save_ops() method
ops1.save_ops(filename)

# load dictionary of operators from object file
ops1.load_ops(filename=filename)

print('Existing dictionary but no object file: {}'.format(ops1.keys()))

### 2.2 With an existing dictionary and object file
Now you can define an operator dictionary for operators you wish to add to the existing operator ops object or initialize a new object with an object file and user defined dictionary

In [ ]:

print(os.getcwd())
tutorial_ops = {
    'unitary1':  np.array([[1, 0], [0, 1]], dtype=complex),
    'unitary2':  0.5*np.array([[complex(1.0, -1.0), complex(1.0, 1.0)], 
        [complex(1.0, 1.0), complex(1.0, -1.0)]], dtype=complex)
}

ops2 = matr.Operator(operators=tutorial_ops, filename=filename)

print('Existing dictionary and object file: {}'.format(ops2.keys()))

### 2.3 With object file but no user defined dictionary

In [ ]:
ops3 = matr.Operator(filename=filename)

print('Object file but no user defined dictionary: {}'.format(ops3.keys()))

### 2.4 With no initial input

In [ ]:
ops4 = matr.Operator()

print('Empty operators dictionary: {}'.format(ops4.keys()))

## 3. Load operator object and Operator class usage examples

In [ ]:
# Load dictionary of operators from the object file
ops1.operators = ops1.load_ops(filename)

# Add new operators to exist operators dictionary
ops1.add_operators(tutorial_ops)

print(ops1.keys())

# Remove no longer needed operators
op_names = ['unitary1','unitary2']

ops1.remove_operators(op_names)

print(ops1.keys())       

### 3.1 Keeping track if operator library is contains only unitary operators

We can check if the operator library contains any non-unitary operators via the object attribute 'is_unitary'. The attribute returns True if the operator library only contains unitary objecrts and False if any non-unitary object exists.

In [ ]:
print(ops1.is_unitary)

Now operators can be added which are non-unitary and we see that the 'is_unitary' attribute gets updated.

In [ ]:
tutorial_ops = {
    'not-unitary1':  np.array([[1, 0], [0, 2]], dtype=complex),
    'not-unitary2':  np.array([[2, 0], [0, 1]], dtype=complex)
}

# Add new operators to exist operators dictionary
ops1.add_operators(tutorial_ops)

# The 'is_unitary' attribute has been changed
print(ops1.is_unitary)

Finally, removing the non-unitary operators we see that the 'is_unitary' attribute is once again updated.  

In [ ]:
# Remove the non-unitary operators
ops1.remove_operators(tutorial_ops)

# Now check if the library attribute 'is_unitary' has changed
print(ops1.is_unitary)

### 3.2 Operators must be complex valued

In [ ]:
tutorial_ops = {
    'not-complex':  np.array([[1, 0], [0, 1]], dtype=int)
}

# Add new operators to exist operators dictionary
ops1.add_operators(tutorial_ops)

### 3.3 Operators must be an array

In [ ]:
tutorial_ops = {
    'not-array':  [[-1, 0], [0, -1]]
}

# Add new operators to exist operators dictionary
ops1.add_operators(tutorial_ops)

### 3.4 Operators **must** be square

Non-square operator arrays with cause a ValueError to be raised. In this example the output ValueError message will read:

In [ ]:
tutorial_ops = {
    'not-square':  np.array([[1, 0, 0], [0, 1, 0]], dtype=complex)
}

try:
    # Add new operators to exist operators dictionary
    ops1.add_operators(tutorial_ops)
except:
    print('ValueError: Operator entry contains a non-square array of size [2,3].')


### 4. Hard coded operators

In the Operator class operator methods can be defined to be called when needed. The operator methods are defined with the following functional form:

``` python
    def <OPERATOR NAME>(self, <input1>, <input2>,..., <inputN>, save=False):

            # Structure for appened names formed from the kwarg keys and values
            def struct(key, val):
                # Ignored: struct and func 
                return f'_{key}{val}'

            # Define a function to construct the operator when needed
            def func(self):
                N_val = N
                k_val = k

                return <user defined function>(N,k)

            # Define variable dictionary for operator
            kwargs = {'<input1>': <input1>, '<input2>': <input2>,...,
                '<inputN>': <inputN>, struct': struct, 'func': func}

            # Check if the operator exist or needs to be saved
            return self.coded_op(save, **kwargs)
```

In the above operator function example, three attributes need to be defined by the user:
1. The operator name must be all capital letters
2. The user must define the structure for appended variable names in the struct function
3. The number of function inputs for the operator function must be defined in the kwargs dictionary

### 4.1 Saving hard coded operators to operator library

When using the hard coded operators they are only computed if they do not exist in the current operator library, otherwise, they are loaded from the library. For high dimensional operators this will save computational resources. If an operator is not in the existing operator library, the user can specify that the operator should be saved with the `save` flag. The defualt is `save=False`.

### 4.2 Using hard coded operator

We can call a hard coded operator as follows:

In [ ]:
new_op = ops3.SIGMA_MINUS(3, 3)
print(new_op)

We see that the operator was not in the existing operator library so it was computed upon the call.

In [ ]:
print(ops3.keys())

Adding the save flag will add the operator to the operator library and then save the library.

In [ ]:
new_op = ops3.SIGMA_MINUS(3, 3, save=True)
print(ops3.keys())

We see that the operator library was updated and that when the operator is called it is loaded from the operator libray rather than being computed.

In [ ]:
ops5 = matr.Operator(filename=filename)
print(ops5.keys())

new_op = ops5.SIGMA_MINUS(3, 3)
print(new_op)